# Manipulasi Data Gym Members Exercise Tracking

Notebook ini menambahkan dua kolom turunan pada `data/gym_members_exercise_tracking.csv`:

- `Tingkat_Aktivitas`
- `Tujuan_Kebugaran`

Jalankan cell dari atas ke bawah saat dataset ingin diproses.

In [12]:
import pandas as pd
from pathlib import Path

In [19]:
DATA_PATH = Path("data/gym_members_exercise_tracking.csv")
OUTPUT_PATH = Path("data/gym_members.csv")

df = pd.read_csv(DATA_PATH)
df.head()

,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI
0,56,Male,88.3,1.71,180,157,60,1.69,1313.0,Yoga,12.6,3.5,4,3,30.20
1,46,Female,74.9,1.53,179,151,66,1.30,883.0,HIIT,33.9,2.1,4,2,32.00
2,32,Female,68.1,1.66,167,122,54,1.11,677.0,Cardio,33.4,2.3,4,2,24.71
3,25,Male,53.2,1.70,190,164,56,0.59,532.0,Strength,28.8,2.1,3,1,18.41
4,38,Male,46.1,1.79,188,158,68,0.64,556.0,Strength,29.2,2.8,3,1,14.39


## 1. Validasi Kolom Wajib

In [20]:
required_columns = [
    "Workout_Frequency (days/week)",
    "Session_Duration (hours)",
    "Calories_Burned",
    "BMI",
    "Fat_Percentage",
]

missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

## 2. Tambah Kolom Tingkat Aktivitas

`Tingkat_Aktivitas` dibuat dari kombinasi frekuensi latihan per minggu, durasi sesi, dan kalori terbakar.

In [21]:
def assign_tingkat_aktivitas(row):
    workout_frequency = row["Workout_Frequency (days/week)"]
    session_duration = row["Session_Duration (hours)"]
    calories_burned = row["Calories_Burned"]

    score = 0

    if workout_frequency <= 1:
        score += 1
    elif workout_frequency <= 3:
        score += 2
    elif workout_frequency <= 5:
        score += 3
    else:
        score += 4

    if session_duration >= 1.5:
        score += 1
    elif session_duration >= 1.0:
        score += 0.5

    if calories_burned >= 900:
        score += 1
    elif calories_burned >= 500:
        score += 0.5

    if score <= 1.5:
        return "Low"
    if score <= 3.0:
        return "Medium"
    if score <= 4.5:
        return "High"
    return "Very High"


df["Activity_Level"] = df.apply(assign_tingkat_aktivitas, axis=1)
df["Activity_Level"].value_counts()

Activity_Level
High         403
Medium       377
Very High    193
Name: count, dtype: int64

## 3. Tambah Kolom Tujuan Kebugaran

`Tujuan_Kebugaran` dibuat dari BMI dan persentase lemak tubuh.

In [22]:
def assign_tujuan_kebugaran(row):
    bmi = row["BMI"]
    fat_percentage = row["Fat_Percentage"]

    if bmi >= 25 or fat_percentage >= 28:
        return "Lose Weight"
    if bmi < 18.5:
        return "Gain Weight"
    return "Maintain Weight"


df["Fitness_Goal"] = df.apply(assign_tujuan_kebugaran, axis=1)
df["Fitness_Goal"].value_counts()

Fitness_Goal
Lose Weight        641
Maintain Weight    235
Gain Weight         97
Name: count, dtype: int64

## 4. Cek Hasil Manipulasi

In [25]:
selected_columns = [
    "Age",
    "Gender",
    "Weight (kg)",
    "Height (m)",
    "Workout_Frequency (days/week)",
    "Session_Duration (hours)",
    "Calories_Burned",
    "Fat_Percentage",
    "BMI",
    "Activity_Level",
    "Fitness_Goal",
]

df[selected_columns].head(10)

,Age,Gender,Weight (kg),Height (m),Workout_Frequency (days/week),Session_Duration (hours),Calories_Burned,Fat_Percentage,BMI,Activity_Level,Fitness_Goal
0,56,Male,88.3,1.71,4,1.69,1313.0,12.6,30.20,Very High,Lose Weight
1,46,Female,74.9,1.53,4,1.30,883.0,33.9,32.00,High,Lose Weight
2,32,Female,68.1,1.66,4,1.11,677.0,33.4,24.71,High,Lose Weight
3,25,Male,53.2,1.70,3,0.59,532.0,28.8,18.41,Medium,Lose Weight
4,38,Male,46.1,1.79,3,0.64,556.0,29.2,14.39,Medium,Lose Weight
5,56,Female,58.0,1.68,5,1.59,1116.0,15.5,20.55,Very High,Maintain Weight
6,36,Male,70.3,1.72,3,1.49,1385.0,21.3,23.76,High,Maintain Weight
7,40,Female,69.7,1.51,3,1.27,895.0,30.6,30.57,Medium,Lose Weight
8,28,Male,121.7,1.94,4,1.03,719.0,28.9,32.34,High,Lose Weight
9,28,Male,101.8,1.84,3,1.08,808.0,29.7,30.07,Medium,Lose Weight


In [24]:
summary = pd.crosstab(df["Tingkat_Aktivitas"], df["Tujuan_Kebugaran"])
summary

KeyError: 'Tingkat_Aktivitas'

## 5. Simpan Dataset Hasil Manipulasi

Hasil disimpan ke file baru agar dataset asli tidak tertimpa.

In [26]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset hasil manipulasi disimpan ke: {OUTPUT_PATH}")

Dataset hasil manipulasi disimpan ke: data/gym_members.csv
